# Analysis
**COSC 301 Project — Malaysia State-Level Socioeconomic & Health Outcomes**

This notebook reads from `data/clean/` (produced by `03_transform.ipynb`) and answers five research questions:

1. **Income & poverty landscape** — Which states are richest and most deprived?
2. **Health infrastructure** — Which states are best and worst served per capita?
3. **Inequality ↔ health correlation** — Does income inequality predict poorer health infrastructure?
4. **Trends over time** — How did income and poverty change between 2019 and 2024?
5. **Long-term trends (1970–2024)** — How has income and poverty evolved over five decades?

**Data sources:** `combined_state.csv` (income, poverty, Gini; 1970–2024) and `health_state.csv` (beds, facilities per capita; 2019/2022/2024 snapshot).

**Note on scope:** W.P. Putrajaya is excluded from the main analysis (`df_main`) as it is an administrative enclave with ~90 000 government-worker residents — not a general-purpose state. W.P. Kuala Lumpur is retained but visually flagged as an urban outlier. W.P. Labuan is classified as East Malaysia (it is geographically in Borneo).

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
from scipy import stats

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent

CLEAN_DIR = PROJECT_ROOT / "data" / "clean"

combined = pd.read_csv(CLEAN_DIR / "combined_state.csv")
hs = pd.read_csv(CLEAN_DIR / "health_state.csv")

# Merged view for cross-sectional analysis (recent survey years with health data)
# territory_type appears in both tables — keep one copy
df = combined[combined["year"].isin([2019, 2022, 2024])].merge(
    hs, on=["state_code", "state", "year"], suffixes=("", "_hs")
)
df = df.drop(columns=[c for c in df.columns if c.endswith("_hs")])

# Convenience: East Malaysia flag (Labuan is in Borneo → East)
EAST = {"Sabah", "Sarawak", "W.P. Labuan"}
df["region"] = df["state"].apply(lambda s: "East" if s in EAST else "West")

# Main analysis dataframe: exclude W.P. Putrajaya
# (administrative enclave with ~90k government-worker population;
#  not a general-purpose state; distorts rankings and correlations)
df_main = df[df["territory_type"] != "admin"].copy()

print("Full df shape    :", df.shape)
print("df_main shape    :", df_main.shape, "(W.P. Putrajaya excluded)")
print("Combined shape   :", combined.shape, "| years:", sorted(combined["year"].unique()))
df_main.head(3)

Full df shape    : (48, 17)
df_main shape    : (45, 17) (W.P. Putrajaya excluded)
Combined shape   : (319, 8) | years: [np.int64(1970), np.int64(1974), np.int64(1976), np.int64(1979), np.int64(1984), np.int64(1987), np.int64(1989), np.int64(1992), np.int64(1995), np.int64(1997), np.int64(1999), np.int64(2002), np.int64(2004), np.int64(2007), np.int64(2009), np.int64(2012), np.int64(2014), np.int64(2016), np.int64(2019), np.int64(2020), np.int64(2022), np.int64(2024)]


,state_code,state,year,territory_type,income_mean,income_median,poverty_absolute,gini,population,hospital_count,facility_count,beds_nonicu,beds_icu,beds_total,beds_per_1000,facilities_per_100k,region
0,JHR,Johor,2019,state,7201.307692,5882.884615,4.623077,0.107621,4009700.0,12,403,3276.0,95.0,3371.0,0.840711,10.050627,West
1,JHR,Johor,2022,state,7529.730769,6215.346154,5.630769,0.108918,4009700.0,12,403,3276.0,95.0,3371.0,0.840711,10.050627,West
2,JHR,Johor,2024,state,8398.769231,7112.923077,3.092308,0.120739,4009700.0,12,403,3276.0,95.0,3371.0,0.840711,10.050627,West


---
## Q1 — Income and Poverty Landscape

> Which states have the highest and lowest household incomes? Which carry the heaviest poverty burden?

In [2]:
# Use 2022 as reference year (most complete survey; 2024 is provisional)
# df_main excludes W.P. Putrajaya; W.P. Kuala Lumpur is retained but flagged
ref = df_main[df_main.year == 2022].copy()

print("=== Income ranking (2022) ===")
income_rank = (
    ref[["state", "income_mean", "income_median", "poverty_absolute", "territory_type"]]
    .sort_values("income_mean", ascending=False)
    .reset_index(drop=True)
)
income_rank.index += 1
income_rank.columns = ["State", "Mean income (RM)", "Median income (RM)", "Poverty rate (%)", "Type"]
income_rank["Mean income (RM)"] = income_rank["Mean income (RM)"].round(0).astype(int)
income_rank["Median income (RM)"] = income_rank["Median income (RM)"].round(0).astype(int)
income_rank["Poverty rate (%)"] = income_rank["Poverty rate (%)"].round(2)
display(income_rank)

=== Income ranking (2022) ===


,State,Mean income (RM),Median income (RM),Poverty rate (%),Type
1,W.P. Kuala Lumpur,13118,10271,1.41,capital
2,Selangor,11359,9146,1.98,state
3,W.P. Labuan,8250,6904,2.50,island_ft
4,Pulau Pinang,8028,6426,2.10,state
5,Melaka,7877,6146,4.50,state
6,Johor,7530,6215,5.63,state
7,Terengganu,7111,5788,6.54,state
8,Negeri Sembilan,6047,4824,5.38,state
9,Sabah,5871,4446,21.82,state
10,Perlis,5625,4730,3.97,state


In [3]:
top3_income = ref.nlargest(3, "income_mean")["state"].tolist()
bot3_income = ref.nsmallest(3, "income_mean")["state"].tolist()
top3_poverty = ref.nlargest(3, "poverty_absolute")["state"].tolist()

print("Highest income states  :", top3_income)
print("Lowest income states   :", bot3_income)
print("Highest poverty states :", top3_poverty)

ratio = ref.income_mean.max() / ref.income_mean.min()
print(f"\nRichest-to-poorest income ratio: {ratio:.1f}x")

Highest income states  : ['W.P. Kuala Lumpur', 'Selangor', 'W.P. Labuan']
Lowest income states   : ['Kelantan', 'Sarawak', 'Kedah']
Highest poverty states : ['Sabah', 'Sarawak', 'Kelantan']

Richest-to-poorest income ratio: 2.7x


In [4]:
print("=== Regional comparison (2022) ===")
regional = (
    ref.groupby("region")[["income_mean", "income_median", "poverty_absolute"]]
    .mean()
    .round(2)
)
display(regional)

# T-test: is the East vs West income gap statistically significant?
east_inc = ref[ref.region == "East"]["income_mean"]
west_inc = ref[ref.region == "West"]["income_mean"]
t_stat, p_val = stats.ttest_ind(east_inc, west_inc)
print(f"\nEast vs West income t-test: t={t_stat:.3f}, p={p_val:.3f}")
print("Significant at α=0.05:", p_val < 0.05)

=== Regional comparison (2022) ===


,income_mean,income_median,poverty_absolute
region,,,
East,6518.19,5221.92,12.69
West,7332.55,5878.89,5.77



East vs West income t-test: t=-0.521, p=0.611
Significant at α=0.05: False


---
## Q2 — Health Infrastructure

> Which states have the most and least hospital beds and facilities per capita?

In [5]:
# Health data is the same across years (one-time snapshot from MoH)
# Use year==2022 reference slice
health_rank = (
    ref[["state", "beds_per_1000", "facilities_per_100k", "hospital_count"]]
    .sort_values("beds_per_1000", ascending=False)
    .reset_index(drop=True)
)
health_rank.index += 1
health_rank.columns = ["State", "Beds / 1 000 pop.", "Facilities / 100 k pop.", "Hospitals"]
health_rank["Beds / 1 000 pop."] = health_rank["Beds / 1 000 pop."].round(3)
health_rank["Facilities / 100 k pop."] = health_rank["Facilities / 100 k pop."].round(1)
display(health_rank)

,State,Beds / 1 000 pop.,Facilities / 100 k pop.,Hospitals
1,Perlis,1.797,14.4,1
2,Sarawak,1.537,12.0,23
3,Perak,1.399,14.7,16
4,Kelantan,1.397,15.1,10
5,Terengganu,1.368,17.1,6
6,Negeri Sembilan,1.368,14.2,8
7,Melaka,1.329,11.2,4
8,Kedah,1.298,14.6,9
9,W.P. Labuan,1.283,14.7,1
10,W.P. Kuala Lumpur,1.219,2.7,7


In [6]:
print("Best-served (beds per 1 000)  :", ref.nlargest(3, "beds_per_1000")["state"].tolist())
print("Worst-served (beds per 1 000) :", ref.nsmallest(3, "beds_per_1000")["state"].tolist())

print("\n=== Regional health infrastructure (2022) ===")
display(ref.groupby("region")[["beds_per_1000", "facilities_per_100k"]].mean().round(3))

Best-served (beds per 1 000)  : ['Perlis', 'Sarawak', 'Perak']
Worst-served (beds per 1 000) : ['Selangor', 'Sabah', 'Johor']

=== Regional health infrastructure (2022) ===


,beds_per_1000,facilities_per_100k
region,,
East,1.179,12.362
West,1.186,12.169


---
## Q3 — Inequality ↔ Health Infrastructure

> Does income inequality (inter-constituency Gini) or poverty rate correlate with fewer health resources per capita?

In [7]:
corr_cols = ["income_mean", "income_median", "poverty_absolute", "gini",
             "beds_per_1000", "facilities_per_100k"]

corr_matrix = ref[corr_cols].corr(method="pearson").round(3)

print("Pearson correlation matrix (2022):")
display(corr_matrix)

Pearson correlation matrix (2022):


,income_mean,income_median,poverty_absolute,gini,beds_per_1000,facilities_per_100k
income_mean,1.000,0.996,-0.603,0.054,-0.397,-0.778
income_median,0.996,1.000,-0.649,0.009,-0.395,-0.746
poverty_absolute,-0.603,-0.649,1.000,0.409,-0.008,0.262
gini,0.054,0.009,0.409,1.000,-0.336,-0.416
beds_per_1000,-0.397,-0.395,-0.008,-0.336,1.000,0.399
facilities_per_100k,-0.778,-0.746,0.262,-0.416,0.399,1.000


In [8]:
# Spearman (rank-based) correlations with beds_per_1000
predictors = ["income_mean", "poverty_absolute", "gini"]
print("Spearman correlations with beds_per_1000:")
for col in predictors:
    r, p = stats.spearmanr(ref[col].dropna(), ref.loc[ref[col].notna(), "beds_per_1000"])
    sig = "*" if p < 0.05 else ""
    print(f"  {col:25s}  r={r:+.3f}  p={p:.3f} {sig}")

print("\nSpearman correlations with facilities_per_100k:")
for col in predictors:
    r, p = stats.spearmanr(ref[col].dropna(), ref.loc[ref[col].notna(), "facilities_per_100k"])
    sig = "*" if p < 0.05 else ""
    print(f"  {col:25s}  r={r:+.3f}  p={p:.3f} {sig}")

Spearman correlations with beds_per_1000:
  income_mean                r=-0.557  p=0.031 *
  poverty_absolute           r=+0.307  p=0.265 
  gini                       r=-0.286  p=0.302 

Spearman correlations with facilities_per_100k:
  income_mean                r=-0.600  p=0.018 *
  poverty_absolute           r=+0.468  p=0.079 
  gini                       r=-0.568  p=0.027 *


In [9]:
# Simple OLS: beds_per_1000 ~ income_mean
mask = ref["income_mean"].notna() & ref["beds_per_1000"].notna()
slope, intercept, r, p, se = stats.linregress(ref.loc[mask, "income_mean"], ref.loc[mask, "beds_per_1000"])
print("OLS: beds_per_1000 ~ income_mean")
print(f"  slope     = {slope:.6f}")
print(f"  intercept = {intercept:.4f}")
print(f"  R²        = {r**2:.3f}")
print(f"  p-value   = {p:.4f}")

OLS: beds_per_1000 ~ income_mean
  slope     = -0.000059
  intercept = 1.6092
  R²        = 0.157
  p-value   = 0.1430


---
## Q4 — Trends Over Time (2019 → 2024)

> Did income and poverty improve across all states between survey years?

In [10]:
se = df_main.copy()

trend = (
    se[se.year.isin([2019, 2022, 2024])]
    .pivot_table(index="state", columns="year", values="income_median")
    .round(0)
    .astype(int)
)
trend["Δ 2019–2024"] = trend[2024] - trend[2019]
trend["% change"] = ((trend[2024] / trend[2019] - 1) * 100).round(1)
trend = trend.sort_values("% change", ascending=False)

print("Median income (RM) by state and year:")
display(trend)

Median income (RM) by state and year:


year,2019,2022,2024,Δ 2019–2024,% change
state,,,,,
Selangor,7663,9146,9822,2159,28.2
Johor,5883,6215,7113,1230,20.9
Terengganu,5462,5788,6574,1112,20.4
Sarawak,3989,4316,4738,749,18.8
Pulau Pinang,6163,6426,7224,1061,17.2
Melaka,5909,6146,6894,985,16.7
Sabah,4104,4446,4704,600,14.6
Kelantan,3566,3698,4084,518,14.5
Negeri Sembilan,4604,4824,5144,540,11.7


In [11]:
pov_trend = (
    se[se.year.isin([2019, 2022, 2024])]
    .pivot_table(index="state", columns="year", values="poverty_absolute")
    .round(2)
)
pov_trend["Δ 2019–2024"] = (pov_trend[2024] - pov_trend[2019]).round(2)
pov_trend = pov_trend.sort_values("Δ 2019–2024")

print("Poverty rate (%) by state and year — sorted by largest reduction:")
display(pov_trend)

Poverty rate (%) by state and year — sorted by largest reduction:


year,2019,2022,2024,Δ 2019–2024
state,,,,
Sabah,21.72,21.82,19.28,-2.44
Johor,4.62,5.63,3.09,-1.53
Terengganu,6.38,6.54,5.03,-1.35
Kelantan,12.82,13.49,11.53,-1.29
W.P. Labuan,3.10,2.50,2.00,-1.10
Sarawak,11.83,13.74,10.90,-0.93
Melaka,3.98,4.50,3.10,-0.88
Selangor,1.86,1.98,1.03,-0.83
Kedah,8.54,8.35,8.13,-0.41


In [12]:
national = (
    se.groupby("year")[["income_mean", "income_median", "poverty_absolute"]]
    .mean()
    .round(2)
)
print("National averages (unweighted mean across states):")
display(national)

National averages (unweighted mean across states):


,income_mean,income_median,poverty_absolute
year,,,
2019,6918.11,5468.25,6.59
2022,7169.68,5747.49,7.16
2024,7682.35,6239.14,6.02


---
## Summary of Findings

Fill in the cells below after running the analysis. These become the basis for the written report.

**Q1 — Income & Poverty**  
*(e.g., W.P. Kuala Lumpur and Selangor lead in household income; Kelantan and Sabah trail significantly; richest-to-poorest income ratio is ~X×)*

**Q2 — Health Infrastructure**  
*(e.g., Which state has the most beds per capita? Which is most underserved?)*

**Q3 — Inequality ↔ Health Correlation**  
*(e.g., Spearman r between income_mean and beds_per_1000 is X, p=Y — [significant / not significant])*

**Q4 — Trends**  
*(e.g., Income rose in all states 2019–2024; poverty fell most sharply in X; Y bucked the trend)*

---
## Q5 — Long-term Trends (1970–2024)

> How has household income and poverty evolved across Malaysian states over five decades?

Uses `combined_state`, which spans DOSM HIES state-level aggregates (1970–2016, 2020) and constituency-derived figures (2019, 2022, 2024).  
Note the gap at 2020–2021: no survey was conducted during the COVID-19 pandemic. 2024 data is provisional.

**Key differences from Q4:** Q4 focuses on the 2019–2024 cross-section with health data; Q5 covers the full historical span.

In [13]:
# Long-term trend: use combined (all survey years, excl. W.P. Putrajaya)
hist_main = combined[combined["territory_type"] != "admin"].copy()

national_hist = (
    hist_main.groupby("year")[["income_mean", "income_median", "poverty_absolute"]]
    .mean()
    .round(2)
)
print("National averages (unweighted, excl. W.P. Putrajaya) — all survey years:")
display(national_hist)

National averages (unweighted, excl. W.P. Putrajaya) — all survey years:


,income_mean,income_median,poverty_absolute
year,,,
1970,244.91,NaN,52.93
1974,341.45,214.55,NaN
1976,505.14,317.64,45.74
1979,619.08,418.62,36.15
1984,1049.14,716.14,21.95
1987,1020.71,728.36,20.83
1989,1110.71,812.64,17.21
1992,1465.86,1054.21,13.74
1995,1826.57,1337.86,9.91


In [14]:
# Poverty reduction: compare 1970 vs 2022 by state
pov_hist = hist_main.pivot_table(index="state", columns="year", values="poverty_absolute")
available_years = sorted(pov_hist.columns)
first_year = available_years[0]   # 1970 for most states
last_year  = available_years[-1]  # 2022

pov_reduction = pov_hist[[first_year, last_year]].copy()
pov_reduction.columns = [f"Poverty {first_year} (%)", f"Poverty {last_year} (%)"]
pov_reduction["Reduction (pp)"] = (
    pov_reduction[f"Poverty {first_year} (%)"] - pov_reduction[f"Poverty {last_year} (%)"]
).round(1)
pov_reduction = pov_reduction.sort_values("Reduction (pp)", ascending=False)

print(f"Poverty headcount ratio: {first_year} → {last_year}")
display(pov_reduction.round(2))

Poverty headcount ratio: 1970 → 2024


,Poverty 1970 (%),Poverty 2024 (%),Reduction (pp)
state,,,
Perlis,73.9,3.63,70.3
Kelantan,76.1,11.53,64.6
Terengganu,68.9,5.03,63.9
Kedah,63.2,8.13,55.1
Johor,45.7,3.09,42.6
Melaka,44.9,3.10,41.8
Pulau Pinang,43.7,2.01,41.7
Perak,48.6,8.37,40.2
Negeri Sembilan,44.8,4.92,39.9
